# Predictive Classification using Random Forest Ensembles
### AIML Minor Project: Breast Cancer Diagnosis (Malignant vs Benign)

**Dataset:** Breast Cancer Wisconsin (Diagnostic) Data Set (Kaggle: `uciml/breast-cancer-wisconsin-data`)
**Algorithm:** `RandomForestClassifier` (scikit-learn)
**Target class:** `diagnosis`, where **M = Malignant (positive class, 1)** and **B = Benign (negative class, 0)**

**Why this dataset suits Random Forest:** it is a tabular, numeric, binary-classification problem with 30 correlated features, only a modest number of rows (569), and a mild class imbalance. Random Forest handles correlated features and non-linear boundaries well, needs no feature scaling, and gives built-in feature importances.

**Workflow (follows the brief):**
1. Phase I: Exploratory Data Analysis (distributions, class imbalance, correlation heatmap)
2. Phase II: Preprocessing (missing values, encoding, 80:20 split, scaling decision)
3. Phase III: Model development (hyperparameter justification for `n_estimators` and `max_depth`)
4. Evaluation: confusion-matrix heatmap, precision and recall, F1, ROC-AUC, feature importance
5. Conclusion: strengths, weaknesses, improvements

> **How to run:** place Kaggle's `data.csv` next to this notebook. If the file is not found, the notebook automatically loads the identical dataset that ships with scikit-learn (same 569 rows, 30 features) and rebuilds Kaggle's column layout, so everything below runs offline too.

## 0. Setup

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score,
                             roc_curve, precision_recall_curve, average_precision_score)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42                      # fixed seed so every result is reproducible
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["savefig.dpi"] = 300      # high-resolution plots for submission
plt.rcParams["figure.dpi"] = 100
os.makedirs("plots", exist_ok=True)
results = {}                           # everything we report is collected here

def save_fig(name):
    """Save the current figure as a 300-dpi PNG in ./plots and show it."""
    plt.tight_layout()
    plt.savefig(f"plots/{name}.png", bbox_inches="tight")
    plt.show()

## 1. Load the data

In [ ]:
def load_data():
    """Load Kaggle's data.csv if present; otherwise rebuild the same table from scikit-learn."""
    for path in ["data.csv", "breast-cancer-wisconsin-data.csv",
                 "/kaggle/input/breast-cancer-wisconsin-data/data.csv"]:
        if os.path.exists(path):
            return pd.read_csv(path), f"Kaggle CSV ({path})"
    from sklearn.datasets import load_breast_cancer
    bc = load_breast_cancer(as_frame=True)
    df = bc.frame.drop(columns="target")
    # rename "mean radius" -> "radius_mean", "radius error" -> "radius_se", "worst radius" -> "radius_worst"
    def kaggle_name(c):
        if c.startswith("mean "):  return c[5:] + "_mean"
        if c.endswith(" error"):   return c[:-6] + "_se"
        if c.startswith("worst "): return c[6:] + "_worst"
    df.columns = [kaggle_name(c) for c in df.columns]
    df.insert(0, "diagnosis", np.where(bc.target == 0, "M", "B"))   # sklearn: 0 = malignant
    df.insert(0, "id", np.arange(1, len(df) + 1))
    df["Unnamed: 32"] = np.nan          # Kaggle's CSV ships with this empty trailing column
    return df, "scikit-learn copy of the same dataset"

df, source = load_data()
print("Data source :", source)
print("Shape       :", df.shape)
df.head()

# Phase I: Exploratory Data Analysis (EDA)
## 2.1 Structure and summary statistics

In [ ]:
df.info()
print()
df.describe().T.round(3).head(12)

## 2.2 Class distribution and class imbalance
The brief asks us to check for class imbalance, because accuracy alone becomes misleading when one class dominates.

In [ ]:
counts = df["diagnosis"].value_counts()
pct = (counts / len(df) * 100).round(2)
print(pd.DataFrame({"count": counts, "percent": pct}))
imbalance_ratio = counts.max() / counts.min()
print(f"\nImbalance ratio (majority:minority) = {imbalance_ratio:.2f} : 1")
print(f"A model that always predicts 'B' would already score {pct['B']:.1f}% accuracy, so we must look beyond accuracy.")

results["class_counts"] = {"B": int(counts["B"]), "M": int(counts["M"])}
results["class_percent"] = {"B": float(pct["B"]), "M": float(pct["M"])}
results["imbalance_ratio"] = float(imbalance_ratio)
results["majority_baseline_accuracy"] = float(pct["B"] / 100)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=df, x="diagnosis", order=["B", "M"], palette=["#4C9F70", "#D1495B"], ax=ax[0])
for p in ax[0].patches:
    ax[0].annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                   ha="center", va="bottom", fontweight="bold")
ax[0].set_title("Class counts"); ax[0].set_xlabel("Diagnosis (B = Benign, M = Malignant)")
ax[1].pie(counts[["B", "M"]], labels=["Benign", "Malignant"], autopct="%1.1f%%",
          colors=["#4C9F70", "#D1495B"], startangle=90, wedgeprops={"edgecolor": "white"})
ax[1].set_title("Class share")
save_fig("01_class_distribution")

## 2.3 Feature distributions by class
The ten `_mean` features, split by diagnosis. Good separation between the two histograms signals a useful predictor.

In [ ]:
mean_cols = [c for c in df.columns if c.endswith("_mean")]
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, col in zip(axes.ravel(), mean_cols):
    sns.histplot(data=df, x=col, hue="diagnosis", hue_order=["B", "M"], kde=True, bins=30,
                 palette=["#4C9F70", "#D1495B"], stat="density", common_norm=False, ax=ax)
    ax.set_title(col)
    ax.set_ylabel("")
fig.suptitle("Distribution of mean features by diagnosis", fontsize=16, y=1.02)
save_fig("02_feature_distributions")

# Skewness quick check (tree models do not need transformations, but it is good EDA)
skew = df[mean_cols].skew().sort_values(ascending=False).round(2)
print("Skewness of mean features:\n", skew)

## 2.4 Correlation heatmap
The brief requires a correlation heatmap before modelling. We plot all 30 features plus the encoded target.

In [ ]:
eda = df.drop(columns=["id", "Unnamed: 32"], errors="ignore").copy()
eda["diagnosis_code"] = (eda["diagnosis"] == "M").astype(int)
corr = eda.drop(columns="diagnosis").corr()

plt.figure(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            linewidths=0.3, cbar_kws={"shrink": 0.7, "label": "Pearson correlation"})
plt.title("Correlation heatmap of all features and the target (diagnosis_code)", fontsize=15)
save_fig("03_correlation_heatmap")

# Strongest linear relationships with the target
target_corr = corr["diagnosis_code"].drop("diagnosis_code").sort_values(ascending=False)
print("Top 8 features positively correlated with malignancy:\n", target_corr.head(8).round(3))

# Multicollinearity: how many feature pairs have |r| > 0.9 ?
feat_corr = corr.drop(index="diagnosis_code", columns="diagnosis_code")
upper = feat_corr.where(np.triu(np.ones(feat_corr.shape, dtype=bool), k=1))
high_pairs = upper.stack()[lambda s: s.abs() > 0.9].sort_values(ascending=False)
print(f"\nFeature pairs with |r| > 0.9: {len(high_pairs)}")
print(high_pairs.head(6).round(3))

results["top_target_corr"] = {k: float(v) for k, v in target_corr.head(5).items()}
results["n_high_corr_pairs"] = int(len(high_pairs))

# Phase II: Preprocessing
## 3.1 Missing / null values

In [ ]:
null_counts = df.isnull().sum()
print("Columns containing nulls:\n", null_counts[null_counts > 0])
print(f"\nTotal null cells: {int(null_counts.sum())}")

# 'Unnamed: 32' is a 100%-empty artefact of the CSV export, and 'id' is only an identifier:
# neither carries information, so both are dropped.
drop_cols = [c for c in ["id", "Unnamed: 32"] if c in df.columns]
data = df.drop(columns=drop_cols)
print("\nDropped columns:", drop_cols)
print("Remaining nulls:", int(data.isnull().sum().sum()))

results["null_cells_raw"] = int(null_counts.sum())
results["dropped_columns"] = drop_cols

## 3.2 Encoding the categorical target
`diagnosis` is the only categorical column. It is binary, so Label Encoding is appropriate (One-Hot would just create two redundant columns). We fix the order explicitly so that **Malignant = 1** is the positive class, which matters for precision, recall and F1.

In [ ]:
le = LabelEncoder().fit(["B", "M"])             # alphabetical: B -> 0, M -> 1
data["diagnosis"] = le.transform(data["diagnosis"])
print("Encoding:", {str(c): int(i) for c, i in zip(le.classes_, le.transform(le.classes_))})

cat_features = data.drop(columns="diagnosis").select_dtypes(include="object").columns.tolist()
print("Other categorical feature columns needing One-Hot:", cat_features or "none (all 30 features are numeric)")

X = data.drop(columns="diagnosis")
y = data["diagnosis"]

## 3.3 Train / test split (80:20, stratified)
`stratify=y` keeps the malignant share identical in both sets, which is important with an imbalanced target.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Malignant share  train: {:.2%}   test: {:.2%}".format(y_train.mean(), y_test.mean()))

# Safety net: median-impute using statistics learnt ONLY from the training set (no leakage).
# This dataset has no nulls after cleaning, so it is a no-op here, but it keeps the pipeline robust.
imputer = SimpleImputer(strategy="median").fit(X_train)
X_train = pd.DataFrame(imputer.transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X.columns, index=X_test.index)

results["n_train"], results["n_test"] = int(len(X_train)), int(len(X_test))
results["n_features"] = int(X.shape[1])

## 3.4 Feature scaling: used or skipped?
Random Forest splits on thresholds of **one feature at a time**, so any monotonic rescaling (what `StandardScaler` does) leaves the order of values, and therefore every split, unchanged. Scaling is not distance-based here, so we **skip it** in the final model.
We verify this claim empirically in Section 5.6 by training the same forest with and without scaling.

# Phase III: Model Development
## 4.1 Baseline forest
A first model with scikit-learn defaults gives a reference point.

In [ ]:
baseline = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)
base_pred = baseline.predict(X_test)
base_prob = baseline.predict_proba(X_test)[:, 1]
print("Baseline (n_estimators=100, max_depth=None)")
print(f"  Accuracy {accuracy_score(y_test, base_pred):.4f} | Recall {recall_score(y_test, base_pred):.4f} | "
      f"F1 {f1_score(y_test, base_pred):.4f} | AUC {roc_auc_score(y_test, base_prob):.4f}")
results["baseline"] = {
    "accuracy": float(accuracy_score(y_test, base_pred)), "precision": float(precision_score(y_test, base_pred)),
    "recall": float(recall_score(y_test, base_pred)), "f1": float(f1_score(y_test, base_pred)),
    "roc_auc": float(roc_auc_score(y_test, base_prob))}

## 4.2 Justifying `n_estimators` and `max_depth`
* **`n_estimators`**: more trees lower the variance of the ensemble, but with diminishing returns and linearly growing cost. We want the point where performance plateaus.
* **`max_depth`**: shallow trees underfit (high bias); unlimited depth can memorise noise (high variance). The ensemble's averaging tames this, but a depth cap can still help generalisation and shrinks the model.

We choose them with **5-fold stratified cross-validation on the training set only**, so the test set stays untouched. The primary score is **F1** (the brief calls it critical for imbalanced data), with recall and ROC-AUC tracked alongside. `max_features` is searched too, since it controls tree diversity.

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 5, 7, 10, None],
    "max_features": ["sqrt", "log2"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid, cv=cv, n_jobs=1, refit="f1", return_train_score=True,
    scoring={"f1": "f1", "recall": "recall", "roc_auc": "roc_auc"})
grid.fit(X_train, y_train)

best = grid.best_params_
print("Best parameters :", best)
print(f"Best CV F1      : {grid.best_score_:.4f}")
cvr = pd.DataFrame(grid.cv_results_)
n_close = int((cvr["mean_test_f1"] >= grid.best_score_ - 0.005).sum())
print(f"{n_close} of {len(cvr)} configurations are within 0.005 F1 of the best, so the score surface is flat near the optimum.")
results["best_params"] = {k: (None if v is None else v) for k, v in best.items()}
results["cv_best_f1"] = float(grid.best_score_)
results["grid_size"] = int(len(cvr))
results["n_close_configs"] = n_close

### Visual evidence for the choice

In [ ]:
cvr["max_depth_label"] = cvr["param_max_depth"].apply(lambda v: "None" if v is None else str(v))
depth_order = ["3", "5", "7", "10", "None"]

fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))

# (a) heatmap: n_estimators x max_depth (best over max_features)
pivot = (cvr.groupby(["max_depth_label", "param_n_estimators"])["mean_test_f1"].max()
            .unstack().loc[depth_order])
sns.heatmap(pivot, annot=True, fmt=".4f", cmap="YlGnBu", ax=ax[0], cbar_kws={"label": "Mean CV F1"})
ax[0].set_title("5-fold CV F1: max_depth vs n_estimators"); ax[0].set_xlabel("n_estimators"); ax[0].set_ylabel("max_depth")

# (b) F1 vs n_estimators with train vs validation (overfitting check) at best depth / max_features
sel = cvr[(cvr["max_depth_label"] == ("None" if best["max_depth"] is None else str(best["max_depth"]))) &
          (cvr["param_max_features"] == best["max_features"])].sort_values("param_n_estimators")
ax[1].plot(sel["param_n_estimators"], sel["mean_train_f1"], "o--", label="Train F1")
ax[1].errorbar(sel["param_n_estimators"], sel["mean_test_f1"], yerr=sel["std_test_f1"],
               fmt="o-", capsize=4, label="CV F1 (mean ± std)")
ax[1].set_title(f"F1 vs n_estimators (max_depth={best['max_depth']}, max_features='{best['max_features']}')")
ax[1].set_xlabel("n_estimators"); ax[1].set_ylabel("F1"); ax[1].legend()
save_fig("04_hyperparameter_tuning")

### Out-of-bag (OOB) error as a second opinion on `n_estimators`
Each tree sees only ~63% of the rows, so the remaining ~37% act as a free validation set. If the OOB error stops falling, extra trees add cost without benefit.

In [ ]:
oob_rf = RandomForestClassifier(max_depth=best["max_depth"], max_features=best["max_features"],
                                oob_score=True, warm_start=True, random_state=RANDOM_STATE, n_jobs=-1)
tree_counts, oob_errors = [], []
for n in range(20, 501, 20):
    oob_rf.set_params(n_estimators=n)
    oob_rf.fit(X_train, y_train)
    tree_counts.append(n); oob_errors.append(1 - oob_rf.oob_score_)

plt.figure(figsize=(8, 4.5))
plt.plot(tree_counts, oob_errors, "o-", color="#2E5EAA")
plt.axvline(best["n_estimators"], color="#D1495B", ls="--", label=f"chosen n_estimators = {best['n_estimators']}")
plt.xlabel("n_estimators"); plt.ylabel("OOB error"); plt.title("Out-of-bag error vs number of trees"); plt.legend()
save_fig("05_oob_error")
results["oob_error_at_chosen"] = float(oob_errors[tree_counts.index(min(tree_counts, key=lambda t: abs(t - best["n_estimators"])))])
results["oob_error_min"] = float(min(oob_errors))

## 4.3 Final model
Trained on the full training split with the cross-validated hyperparameters.

In [ ]:
rf = RandomForestClassifier(**best, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]          # probability of Malignant
print("Final model:", rf)

# Evaluation (required metrics)
## 5.1 Confusion matrix (heatmap)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (benign correctly cleared)        : {tn}")
print(f"False Positives (benign wrongly flagged malignant): {fp}")
print(f"False Negatives (malignant MISSED)                : {fn}")
print(f"True Positives  (malignant correctly caught)      : {tp}")

labels = np.array([[f"TN\n{tn}", f"FP\n{fp}"], [f"FN\n{fn}", f"TP\n{tp}"]])
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", cbar=False, ax=ax[0], annot_kws={"size": 16, "weight": "bold"},
            xticklabels=["Pred Benign", "Pred Malignant"], yticklabels=["Actual Benign", "Actual Malignant"])
ax[0].set_title("Confusion matrix (counts)")
cm_norm = confusion_matrix(y_test, y_pred, normalize="true")
sns.heatmap(cm_norm, annot=True, fmt=".1%", cmap="Greens", cbar=False, ax=ax[1], annot_kws={"size": 16, "weight": "bold"},
            xticklabels=["Pred Benign", "Pred Malignant"], yticklabels=["Actual Benign", "Actual Malignant"])
ax[1].set_title("Confusion matrix (row-normalised)")
save_fig("06_confusion_matrix")

results["confusion"] = {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

## 5.2 Precision, recall and F1
* **Precision** = TP / (TP + FP): of the cases we flag malignant, how many really are (*exactness*).
* **Recall** = TP / (TP + FN): of all truly malignant cases, how many we catch (*completeness*).
* **F1** = 2·P·R / (P + R): harmonic mean, which punishes a model that is strong on only one of the two.

In [ ]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
spec = tn / (tn + fp)
print(f"Accuracy    : {acc:.4f}")
print(f"Precision   : {prec:.4f}")
print(f"Recall      : {rec:.4f}  (sensitivity)")
print(f"Specificity : {spec:.4f}")
print(f"F1-score    : {f1:.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"], digits=4))

results["final"] = {"accuracy": float(acc), "precision": float(prec), "recall": float(rec),
                    "specificity": float(spec), "f1": float(f1)}

### Precision-recall trade-off: moving the decision threshold
The default threshold of 0.5 is not sacred. Lowering it catches more malignant cases (higher recall) at the price of more false alarms (lower precision), and raising it does the opposite. *This table is illustrative only: choosing a threshold on the test set would leak information, so in practice it would be tuned on validation data.*

In [ ]:
rows = []
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    p = (y_prob >= t).astype(int)
    cm_t = confusion_matrix(y_test, p)
    rows.append({"threshold": t, "precision": precision_score(y_test, p), "recall": recall_score(y_test, p),
                 "f1": f1_score(y_test, p), "FP": int(cm_t[0, 1]), "FN": int(cm_t[1, 0])})
thr_table = pd.DataFrame(rows).round(4)
print(thr_table.to_string(index=False))
results["threshold_table"] = thr_table.to_dict(orient="records")

prec_c, rec_c, thr_c = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
results["average_precision"] = float(ap)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
ax[0].plot(thr_c, prec_c[:-1], label="Precision", color="#2E5EAA")
ax[0].plot(thr_c, rec_c[:-1], label="Recall", color="#D1495B")
ax[0].axvline(0.5, color="grey", ls="--", label="default threshold 0.5")
ax[0].set_xlabel("Decision threshold"); ax[0].set_ylabel("Score"); ax[0].set_title("Precision and recall vs threshold"); ax[0].legend()
ax[1].plot(rec_c, prec_c, color="#2E5EAA", lw=2)
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
ax[1].set_title(f"Precision-recall curve (AP = {ap:.4f})")
ax[1].set_ylim(0.0, 1.02)
save_fig("07_precision_recall_tradeoff")

## 5.3 ROC curve and AUC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
fpr_b, tpr_b, _ = roc_curve(y_test, base_prob)
results["final"]["roc_auc"] = float(auc)

plt.figure(figsize=(6.5, 6))
plt.plot(fpr, tpr, lw=2.5, color="#2E5EAA", label=f"Tuned Random Forest (AUC = {auc:.4f})")
plt.plot(fpr_b, tpr_b, lw=1.5, color="#E0A100", ls="-.", label=f"Baseline Random Forest (AUC = {results['baseline']['roc_auc']:.4f})")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random guess (AUC = 0.5)")
plt.fill_between(fpr, tpr, alpha=0.12, color="#2E5EAA")
plt.xlabel("False Positive Rate (FPR)"); plt.ylabel("True Positive Rate (TPR)")
plt.title("ROC curve"); plt.legend(loc="lower right")
save_fig("08_roc_curve")

## 5.4 Feature importance
Impurity-based importance (mean decrease in Gini impurity, averaged over all trees), shown for the top 15 features. Error bars are the standard deviation across trees.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
std = pd.Series(np.std([t.feature_importances_ for t in rf.estimators_], axis=0), index=X.columns)
top = importances.head(15)[::-1]

plt.figure(figsize=(9, 6.5))
plt.barh(top.index, top.values, xerr=std[top.index].values, color="#2E5EAA", ecolor="#999999", capsize=3)
plt.xlabel("Mean decrease in impurity"); plt.title("Top 15 feature importances (Random Forest)")
save_fig("09_feature_importance")

print(importances.head(10).round(4))
print(f"\nTop 5 features account for {importances.head(5).sum():.1%} of total importance; top 10 for {importances.head(10).sum():.1%}.")
results["top_features"] = {k: float(v) for k, v in importances.head(10).items()}
results["top5_share"] = float(importances.head(5).sum())
results["top10_share"] = float(importances.head(10).sum())

### Cross-check with permutation importance
Impurity importance can favour features with many distinct values and splits credit between correlated features. Permutation importance (drop in test F1 when a feature is shuffled) is a model-agnostic second opinion.

In [ ]:
perm = permutation_importance(rf, X_test, y_test, scoring="f1", n_repeats=30, random_state=RANDOM_STATE, n_jobs=-1)
perm_s = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
perm_top = perm_s.head(10)[::-1]
plt.figure(figsize=(8, 5))
plt.barh(perm_top.index, perm_top.values, xerr=pd.Series(perm.importances_std, index=X.columns)[perm_top.index].values,
         color="#D1495B", ecolor="#999999", capsize=3)
plt.xlabel("Mean drop in test F1 when shuffled"); plt.title("Permutation importance (top 10)")
save_fig("10_permutation_importance")
results["perm_top"] = {k: float(v) for k, v in perm_s.head(5).items()}
overlap = set(importances.head(10).index) & set(perm_s.head(10).index)
print("Features in both top-10 lists:", sorted(overlap))
print("Permutation importances are tiny because correlated features (radius/perimeter/area) cover for each other when one is shuffled.")

## 5.5 Robustness: is the test score a fluke?
A single 80:20 split on 114 test rows is noisy (one patient = 0.9 percentage points of recall), so we also run 5-fold stratified CV on the **entire** dataset.

In [ ]:
cv_scores = {}
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    s = cross_val_score(RandomForestClassifier(**best, random_state=RANDOM_STATE, n_jobs=-1),
                        X, y, cv=cv, scoring=metric)
    cv_scores[metric] = (float(s.mean()), float(s.std()))
    print(f"{metric:>10}: {s.mean():.4f} ± {s.std():.4f}")
results["cv_full"] = cv_scores

## 5.6 Was skipping `StandardScaler` justified?
Same hyperparameters and seed, with and without scaling:

In [ ]:
scaled = Pipeline([("scaler", StandardScaler()),
                   ("rf", RandomForestClassifier(**best, random_state=RANDOM_STATE, n_jobs=-1))]).fit(X_train, y_train)
sc_pred = scaled.predict(X_test)
sc_prob = scaled.predict_proba(X_test)[:, 1]
same = bool((sc_pred == y_pred).all())
print(f"Unscaled: F1 {f1:.4f}, AUC {auc:.4f}")
print(f"Scaled  : F1 {f1_score(y_test, sc_pred):.4f}, AUC {roc_auc_score(y_test, sc_prob):.4f}")
print("Identical test predictions:", same)
results["scaling_identical_predictions"] = same
results["scaling_f1"] = float(f1_score(y_test, sc_pred))
results["scaling_auc"] = float(roc_auc_score(y_test, sc_prob))

## 5.7 Potential improvement: Gradient Boosting comparison
The brief suggests Gradient Boosting as a follow-up. A quick, untuned comparison on the same split:

In [ ]:
gb = GradientBoostingClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
gb_pred = gb.predict(X_test); gb_prob = gb.predict_proba(X_test)[:, 1]
comparison = pd.DataFrame({
    "Baseline RF": [accuracy_score(y_test, base_pred), precision_score(y_test, base_pred), recall_score(y_test, base_pred),
                    f1_score(y_test, base_pred), roc_auc_score(y_test, base_prob)],
    "Tuned RF":    [acc, prec, rec, f1, auc],
    "Gradient Boosting (default)": [accuracy_score(y_test, gb_pred), precision_score(y_test, gb_pred), recall_score(y_test, gb_pred),
                                    f1_score(y_test, gb_pred), roc_auc_score(y_test, gb_prob)],
}, index=["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]).round(4)
print(comparison)
results["model_comparison"] = comparison.to_dict()

# Summary

In [ ]:
print("=" * 60)
print("FINAL TEST-SET RESULTS (tuned Random Forest)")
print("=" * 60)
print(f"Hyperparameters : {best}")
print(f"Accuracy {acc:.4f} | Precision {prec:.4f} | Recall {rec:.4f} | F1 {f1:.4f} | ROC-AUC {auc:.4f}")
print(f"Confusion matrix: TN={tn} FP={fp} FN={fn} TP={tp}")

with open("results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print("\nSaved: results.json and 10 high-resolution plots in ./plots")

## Conclusion

**Strengths**
* Random Forest delivers very high accuracy, precision, recall and ROC-AUC on this dataset (exact figures printed above), and cross-validation on the full data confirms the score is not a lucky split.
* It needs almost no preprocessing: no scaling (verified in 5.6), robust to correlated features and skewed distributions, and it provides interpretable feature importances (impurity-based importance points to the size/shape family of features: worst perimeter, area, concave points and radius).
* Hyperparameter tuning showed a flat optimum: many settings of `n_estimators` and `max_depth` perform almost identically, so the model is stable rather than finely balanced.

**Weaknesses**
* The test set is small (114 rows), so each misclassified patient moves recall by roughly one percentage point; confidence intervals are wide.
* In a cancer-screening setting a **false negative (missed malignancy) is far costlier than a false positive**. The default 0.5 threshold does not encode that cost; Section 5.2 shows the trade-off.
* Correlated features (radius, perimeter and area measure nearly the same thing) split importance between them: permutation importance of any single feature is close to zero because its siblings compensate, so importance values should be read as group-level signals.
* On this particular split the tuned forest scored slightly *below* the default forest on the test set (a difference of about two patients), although it scored marginally higher in cross-validation. At n = 114 that gap is within noise, which is a reminder not to over-interpret one split.
* The model is a black box compared with a single tree or logistic regression, and the dataset (one institution, 569 samples) may not generalise to other hospitals or imaging set-ups.

**Possible improvements**
* Tune the decision threshold on a validation set (or use `class_weight`) to prioritise recall.
* Try Gradient Boosting / XGBoost / LightGBM with tuning, and compare with logistic regression and SVM as baselines.
* Use nested cross-validation, calibrate probabilities (`CalibratedClassifierCV`), and add SHAP values for per-patient explanations.
* Validate on an external dataset before any clinical use. This project is an educational exercise, not a diagnostic tool.